# Human Activity Recognition from Video Using CNN + LSTM

**Raj Rishav** · Department of Electronics and Electrical Engineering, IIT Guwahati  
**Dataset:** [Human Activity Recognition - Video Dataset (Kaggle)](https://www.kaggle.com/datasets/sharjeelmazhar/human-activity-recognition-video-dataset)  
**Code:** [GitHub repository](https://github.com/Ahmdullah16/Raj_Rishav)  
**Data:** [RAJ Rishav Google Drive folder](https://drive.google.com/drive/folders/1EIboVuDhhsE_qL27u3u6QrzNyuAhWfbD)

This is the **complete, executable and commented** project. No Kaggle account or new video download is needed: it reads the **132 clips already stored in Drive**. It generates a reproducible, per-class **70%/15%/15% split** (92/20/20 clips), extracts fixed video frame sequences, trains CNN–LSTM models, selects the best model on the validation set, evaluates the held-out test set **once**, and saves model weights, metrics, visualizations and class mapping.

> **Important distinction:** The provided project PDF and predecessor notebook used *UCF101 (101 categories)*. This notebook instead trains on **seven categories** from a **different Kaggle dataset**. Earlier UCF101 accuracy (15.42%) is **not** a seven-class result. No new results are claimed before you run the training cells.

> **November 8 decoder fix:** KaggleHub-provided files carrying `.mp4` names may be ZIP archives containing real MP4s. This notebook detects the archive, safely extracts to local Colab cache, verifies decodability, and uses the actual video for model training. No duplicate footage is added to Drive.


## 1. Runtime setup

Use **Runtime → Change runtime type → GPU (T4)** when available. Dependencies include TensorFlow, OpenCV, NumPy, pandas, scikit-learn and Matplotlib. The notebook can also run on CPU but will be slower. Avoid committing videos, Google Drive credentials or `.keras` checkpoints to a public repository.

In [ ]:
%pip -q install opencv-python-headless pandas scikit-learn matplotlib seaborn

In [ ]:
import os, sys, json, csv, random, platform
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from google.colab import drive

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/RAJ Rishav/HAR_Kaggle_Subset')
RESULTS_ROOT = Path('/content/drive/MyDrive/RAJ Rishav/HAR_Results')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
SEED = 42
SEQ_LENGTH = 20
IMAGE_SIZE = 64
BATCH_SIZE = 8
EPOCHS = 25
# Run the three scratch variants (or optional MobileNetV2) for comparisons.
# For a first run we select the efficient 3-layer model as the project main model.
RUN_VARIANTS = ['cnn_lstm_3']
# Uncomment for a fuller controlled comparison if you have enough Colab GPU time:
# RUN_VARIANTS = ['cnn_lstm_1', 'cnn_lstm_2', 'cnn_lstm_3']
# RUN_VARIANTS += ['mobilenetv2_lstm']  # downloads pretrained weights
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('Python:', platform.python_version(), '| TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Data root:', DATA_ROOT, '| present:', DATA_ROOT.is_dir())
if not DATA_ROOT.is_dir():
    raise FileNotFoundError('Dataset folder is not mounted. Check your Drive path above.')

## 2. Dataset integrity and video decoding fix

**Important discovery (verified against your actual Google Drive file):**  
`Clapping (1).mp4` is **not an MP4 container on disk**. Its bytes start with a ZIP signature, and the ZIP contains a real H.264 MP4 named `Clapping (1).mp4`. Thus, `cv2.VideoCapture()` fails when handed the outer file, despite correct path and file size. Other downloads may have the same issue.

The notebook now:

1. Indexes the seven activity folders using actual files on Drive.
2. **Identifies ZIP-wrapped videos by their file signature** (`zipfile.is_zipfile`), never guessing from the `.mp4` extension.
3. Safely streams the real enclosed video into a temporary Colab runtime cache (`/content/har_unpacked_video_cache`), not into Drive, and verifies the first frame with OpenCV.
4. Excludes and logs unreadable clips before making new **random, stratified 70/15/15 splits**. All seven activity classes must remain represented.
5. Reuses decoded cached MP4s during CNN–LSTM training, avoiding repeat Drive reads. The cache is deleted when Colab runtime resets; the original Drive files are unchanged.

**Existing Google Drive videos should not be renamed or deleted.**


In [ ]:
# Full implementation embedded from src/har/data.py
"""Dataset indexing and deterministic, per-activity 70/15/15 splitting.

No videos are duplicated, moved or modified. A manifest references each source video.
"""
from __future__ import annotations

import csv
import json
import random
from collections import Counter
from pathlib import Path
from typing import Iterable

LABELS = (
    "clapping", "meet_and_split", "sitting", "standing_still", "walking",
    "walking_while_reading_book", "walking_while_using_phone",
)
DISPLAY_NAMES = {
    "clapping": "Clapping",
    "meet_and_split": "Meet and Split",
    "sitting": "Sitting",
    "standing_still": "Standing Still",
    "walking": "Walking",
    "walking_while_reading_book": "Walking While Reading Book",
    "walking_while_using_phone": "Walking While Using Phone",
}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".mpeg", ".mpg"}
SPLITS = ("train", "val", "test")
FRACTIONS = (0.70, 0.15, 0.15)


def label_from_path(value: str) -> str:
    """Classify by the immediate parent folder (not partial filename matches)."""
    from pathlib import PurePosixPath
    parent = PurePosixPath(value.replace("\\", "/")).parent.name
    clean = "_".join(parent.lower().replace("-", " ").split())
    aliases = {"meet_and_split": "meet_and_split"}
    clean = aliases.get(clean, clean)
    if clean not in LABELS:
        raise ValueError(f"Unknown activity folder {parent!r} for {value!r}")
    return clean


def read_manifest(path: str | Path) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise ValueError("Manifest is empty")
    checked = []
    seen = set()
    for row in rows:
        p = row.get("video_path", "").replace("\\", "/")
        if not p or p.startswith("/") or ".." in Path(p).parts or Path(p).suffix.lower() not in VIDEO_EXTENSIONS:
            raise ValueError(f"Invalid video path: {p}")
        if p in seen:
            raise ValueError(f"Duplicate source path: {p}")
        seen.add(p)
        label = label_from_path(p)
        if row.get("activity") and row["activity"] != label:
            raise ValueError(f"Label mismatch: {p}: {row['activity']} != {label}")
        size = int(row.get("bytes") or 0)
        if size <= 0:
            raise ValueError(f"Invalid byte size for {p}")
        checked.append({"video_path": p, "activity": label, "bytes": size})
    return checked


def index_directory(root: str | Path) -> list[dict]:
    """Read an extracted Kaggle folder, retaining paths relative to root."""
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Video directory not found: {root}")
    rows = []
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS:
            rel = path.relative_to(root).as_posix()
            try:
                label = label_from_path(rel)
            except ValueError:
                continue  # reject unrelated video folders instead of guessing labels
            if path.stat().st_size:
                rows.append({"video_path": rel, "activity": label, "bytes": path.stat().st_size})
    if not rows:
        raise ValueError("No labeled videos found in the directory")
    return rows


def stratified_split(rows: Iterable[dict], seed: int = 42) -> list[dict]:
    """Random per-class split, close to 70/15/15 with exact global quotas.

    The fixed seed makes the video-to-split allocation reproducible. All three
    splits contain each class when class size >= 3. The final target totals
    follow largest-remainder rounding across the full dataset.
    """
    items = [dict(r) for r in rows]
    if not items:
        raise ValueError("Cannot split an empty dataset")
    seen = [r["video_path"] for r in items]
    if len(set(seen)) != len(seen):
        raise ValueError("Duplicate videos in manifest")
    grouped: dict[str, list[dict]] = {}
    for row in items:
        grouped.setdefault(row["activity"], []).append(row)
    if any(len(group) < 3 for group in grouped.values()):
        raise ValueError("Each category requires >=3 videos for three nonempty splits")
    rng = random.Random(seed)
    labels = sorted(grouped)
    for label in labels:
        grouped[label].sort(key=lambda r: r["video_path"])
        rng.shuffle(grouped[label])

    n = len(items)
    raw_totals = [n * f for f in FRACTIONS]
    totals = [int(x) for x in raw_totals]
    for i in sorted(range(3), key=lambda i: (-(raw_totals[i] - totals[i]), i))[:n - sum(totals)]:
        totals[i] += 1

    counts = {}
    for label in labels:
        m = len(grouped[label])
        ideal = [m * f for f in FRACTIONS]
        quota = [int(x) for x in ideal]
        for i in sorted(range(3), key=lambda i: (-(ideal[i] - quota[i]), i))[:m - sum(quota)]:
            quota[i] += 1
        counts[label] = quota

    current = [sum(counts[label][i] for label in labels) for i in range(3)]
    # Rebalance per-class counts to globally rounded target, minimizing error
    # versus each category's ideal fractional quotas.
    while current != totals:
        donors = [i for i in range(3) if current[i] > totals[i]]
        receivers = [i for i in range(3) if current[i] < totals[i]]
        choices = []
        for donor in donors:
            for receiver in receivers:
                for label in labels:
                    quota = counts[label]
                    if quota[donor] <= 1:
                        continue
                    ideal = [len(grouped[label]) * f for f in FRACTIONS]
                    before = sum((quota[j] - ideal[j])**2 for j in (donor, receiver))
                    after = ((quota[donor]-1 - ideal[donor])**2
                             + (quota[receiver]+1 - ideal[receiver])**2)
                    choices.append((round(after - before, 9), label, donor, receiver))
        if not choices:
            raise RuntimeError("Could not satisfy target quotas")
        _, label, donor, receiver = min(choices)
        counts[label][donor] -= 1
        counts[label][receiver] += 1
        current[donor] -= 1
        current[receiver] += 1

    result = []
    for label in labels:
        offset = 0
        for split, count in zip(SPLITS, counts[label]):
            for row in grouped[label][offset:offset + count]:
                result.append({**row, "split": split})
            offset += count
    assert len(result) == len(items)
    assert len({r["video_path"] for r in result}) == len(items)
    assert tuple(Counter(r["split"] for r in result)[s] for s in SPLITS) == tuple(totals)
    return sorted(result, key=lambda r: (r["activity"], r["video_path"]))


def write_csv(path: str | Path, rows: list[dict], columns: tuple[str, ...] | list[str]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, "") for k in columns})


def summary(rows: list[dict], seed: int) -> dict:
    return {
        "source": "Kaggle: sharjeelmazhar/human-activity-recognition-video-dataset",
        "dataset_is_not_ucf101": True,
        "seed": seed,
        "video_count": len(rows),
        "total_video_bytes": sum(int(r["bytes"]) for r in rows),
        "classes": sorted(set(r["activity"] for r in rows)),
        "splits": {split: sum(r["split"] == split for r in rows) for split in SPLITS},
        "counts_by_class_and_split": {
            label: {split: sum(r["activity"] == label and r["split"] == split for r in rows)
                    for split in SPLITS}
            for label in sorted(set(r["activity"] for r in rows))
        },
        "warning": "Random video-level splits can leak actors/scenes. Use group-based splits where group IDs exist.",
        "results_note": "No model training/evaluation metrics are implied by this split.",
    }

In [ ]:
"""Robust video decoding for Kaggle HAR assets stored on Google Drive.

Some KaggleHub per-file downloads arrived as ZIP *archives* saved with an .mp4
extension. They are not directly decodable by OpenCV. This module detects the
actual file signature, safely extracts an enclosed MP4 to Colab's local cache,
and then samples RGB frames without modifying or duplicating the Drive data.
"""
from __future__ import annotations

import hashlib
import os
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np

VIDEO_SUFFIXES = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v', '.mpg', '.mpeg'}
MAX_EXTRACTED_VIDEO_BYTES = 1_000_000_000  # safety cap per clip, not whole dataset


def resolve_video_file(path: str | Path, cache_dir: str | Path | None = None) -> Path:
    """Return real video bytes, unwrapping ZIP-disguised MP4s when necessary.

    We stream exactly one video member from each ZIP archive to a deterministic
    *local* temp path (not Drive). Safe from path traversal because the archive
    member path is never used as the extraction destination. The original Drive
    file remains untouched. Files already stored as real videos pass through.
    """
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f'Video not found: {source}')
    if not zipfile.is_zipfile(source):
        return source

    cache = Path(cache_dir or os.environ.get('HAR_VIDEO_CACHE',
                       Path(tempfile.gettempdir()) / 'har_unpacked_videos'))
    cache.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        candidates = [item for item in archive.infolist()
                      if not item.is_dir() and Path(item.filename).suffix.lower() in VIDEO_SUFFIXES]
        if len(candidates) != 1:
            raise ValueError(
                f'{source.name} is a ZIP archive with {len(candidates)} video members; '
                'expected exactly one video, cannot select safely.')
        member = candidates[0]
        if not (0 < member.file_size <= MAX_EXTRACTED_VIDEO_BYTES):
            raise ValueError(f'Invalid/unreasonably large video inside {source}: '
                             f'{member.file_size} bytes')
        if member.flag_bits & 0x1:
            raise ValueError(f'Encrypted ZIP video not supported: {source}')
        stat = source.stat()
        identity = f'{source.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|{member.CRC}'
        digest = hashlib.sha256(identity.encode('utf-8')).hexdigest()[:24]
        output = cache / f'{digest}{Path(member.filename).suffix.lower()}'
        if output.is_file() and output.stat().st_size == member.file_size:
            return output
        partial = output.with_suffix(output.suffix + '.part')
        try:
            with archive.open(member, 'r') as incoming, partial.open('wb') as outgoing:
                # zipfile's reader verifies CRC as it reaches EOF.
                shutil.copyfileobj(incoming, outgoing, length=1024 * 1024)
            if partial.stat().st_size != member.file_size:
                raise ValueError(f'Extracted file size mismatch: {source.name}')
            partial.replace(output)
        except BaseException:
            partial.unlink(missing_ok=True)
            raise
        return output


def validate_video(path: str | Path, cache_dir: str | Path | None = None) -> dict:
    """Verify readable first frame and return helpful diagnostics."""
    import cv2
    extracted = resolve_video_file(path, cache_dir=cache_dir)
    cap = cv2.VideoCapture(str(extracted))
    try:
        if not cap.isOpened():
            raise ValueError(f'OpenCV cannot decode: {extracted} (original {path})')
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        ok, frame = cap.read()
        if not ok or frame is None or frame.size == 0:
            raise ValueError(f'Cannot read first frame from: {extracted}')
        return {'frame_count_reported': count,
                'height': int(frame.shape[0]), 'width': int(frame.shape[1]),
                'was_zip': str(Path(path)) != str(extracted)}
    finally:
        cap.release()


def sample_video(path: str | Path, seq_length: int = 20, image_size: int = 64) -> np.ndarray:
    """Return (T,H,W,3) normalized RGB frames, repeating if source is short."""
    import cv2
    extracted = resolve_video_file(path)
    cap = cv2.VideoCapture(str(extracted))
    try:
        if not cap.isOpened():
            raise ValueError(f'Cannot open video: {extracted} (source: {path})')
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if n <= 0:
            # Some decoders don't expose frame_count; try sequential fallback.
            frames = []
            while len(frames) < seq_length:
                ok, frame = cap.read()
                if not ok:
                    break
                frames.append(frame)
            if not frames:
                raise ValueError(f'Empty or undecodable video: {path}')
            while len(frames) < seq_length:
                frames.append(frames[-1])
        else:
            indices = np.linspace(0, n - 1, seq_length).round().astype(int)
            frames = []
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
                ok, frame = cap.read()
                if not ok or frame is None:
                    raise ValueError(f'Failed to read frame {idx} from {path}')
                frames.append(frame)
        frames = [cv2.cvtColor(cv2.resize(frame, (image_size, image_size)), cv2.COLOR_BGR2RGB)
                  for frame in frames]
        return np.asarray(frames, dtype=np.float32) / np.float32(255.0)
    finally:
        cap.release()


# Dataset integrity and actual video decoding preflight.
# IMPORTANT: some files ending in .mp4 are really ZIP archives containing MP4s.
# We extract only to temporary Colab disk, never change the Drive originals.
import shutil
from tqdm.auto import tqdm
VIDEO_CACHE = Path('/content/har_unpacked_video_cache')
os.environ['HAR_VIDEO_CACHE'] = str(VIDEO_CACHE)
rows = index_directory(DATA_ROOT)
print('Located', len(rows), 'labeled source files,', round(sum(r['bytes'] for r in rows)/1e9,3), 'GB on Drive')
zip_count = sum(zipfile.is_zipfile(DATA_ROOT / r['video_path']) for r in rows)
print(f'Files containing ZIP archives despite video filenames: {zip_count} / {len(rows)}')
if zip_count:
    free = shutil.disk_usage('/content').free
    required = sum(r['bytes'] for r in rows if zipfile.is_zipfile(DATA_ROOT / r['video_path']))
    print('Colab temporary free storage:', round(free/1e9, 2), 'GB')
    if free < required + 250_000_000:
        raise RuntimeError(
            'Not enough temporary Colab disk to unpack the ZIP-wrapped videos. '
            'Free Colab runtime disk space or run fewer videos; DO NOT delete Drive originals.')

# Check each real video can open and decode its first frame BEFORE computing splits.
# Failing clips are recorded, not used for training or inflated in the split.
verified_rows = []
bad_videos = []
for entry in tqdm(rows, desc='Validate and unpack source videos'):
    path = DATA_ROOT / entry['video_path']
    try:
        info = validate_video(path, cache_dir=VIDEO_CACHE)
        verified_rows.append(entry)
    except Exception as exc:
        bad_videos.append({'path': entry['video_path'], 'reason': str(exc)[:500]})
if bad_videos:
    (RESULTS_ROOT/'unreadable_videos.json').write_text(json.dumps(bad_videos, indent=2))
    print('WARNING:', len(bad_videos), 'unreadable source files. Details in HAR_Results/unreadable_videos.json')
    print('First errors:', bad_videos[:3])
rows = verified_rows
if not rows:
    raise RuntimeError('None of the source videos can be decoded. Check Google Drive mount and files.')
actual_by_class = Counter(r['activity'] for r in rows)
print('Validated videos by class:', dict(sorted(actual_by_class.items())))
if set(actual_by_class) != set(LABELS):
    raise RuntimeError('Valid dataset must contain all seven classes. Decodable classes: '
                       + str(sorted(actual_by_class)))
if min(actual_by_class.values()) < 3:
    raise RuntimeError('Each activity needs at least 3 decodable videos for train, val, and test.')

# Regenerate a stratified 70/15/15 split *only from verified physical videos*.
# The output CSV is also retained under the Drive dataset for future runs.
split_rows = stratified_split(rows, seed=SEED)
split_path = DATA_ROOT / 'train_val_test_splits_70_15_15.csv'
write_csv(split_path, split_rows, ('video_path','activity','bytes','split'))
summary_data = summary(split_rows, seed=SEED)
summary_data['zip_wrapped_video_count'] = int(zip_count)
summary_data['unreadable_video_count'] = len(bad_videos)
(DATA_ROOT/'dataset_summary_70_15_15.json').write_text(json.dumps(summary_data, indent=2))
print('Saved verified splits to:', split_path)
print('Actual split counts:', summary_data['splits'])
print(pd.crosstab(pd.DataFrame(split_rows).activity, pd.DataFrame(split_rows).split).to_string())
assert len({r['video_path'] for r in split_rows}) == len(split_rows)
assert all((DATA_ROOT / r['video_path']).is_file() for r in split_rows)


## 3. Data exploration and class distribution

Count video clips per activity and per split, and inspect selected sample frames. These figures summarize the real files in Drive, not synthetic data.

In [ ]:
df = pd.DataFrame(split_rows)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
pd.crosstab(df.activity, df.split)[['train','val','test']].plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set(title='Videos by activity and split', ylabel='Number of videos', xlabel='Activity')
axes[0].tick_params(axis='x', labelrotation=40)
axes[1].bar(['train','val','test'], [sum(df.split==s) for s in ('train','val','test')])
axes[1].set(title='Overall 70/15/15 split', ylabel='Video count')
plt.tight_layout()
fig.savefig(RESULTS_ROOT/'dataset_distribution.png', dpi=180)
plt.show()

## 4. Video preprocessing (20-frame RGB tensor)

For a source clip with \(F\) frames, uniformly sample \(T=20\) frame indices \(t_k=\operatorname{round}(k(F-1)/(T-1))\). Resize every frame to \(64\times64\), convert OpenCV BGR to RGB, and scale to \([0,1]\): \(X\in[0,1]^{20\times64\times64\times3}\).

**ZIP-wrap fix:** before decoding, `sample_video` calls `resolve_video_file` to extract a real MP4 from any misleadingly named ZIP archive. This was confirmed on your Drive's `Clapping (1).mp4` video. Files already in actual video format are decoded directly.


In [ ]:
# The ZIP-aware functions `resolve_video_file`, `validate_video` and
# `sample_video` were defined during the dataset preflight (Section 2).
# You may check the source and its cached decoded MP4 files here.
print('ZIP-wrapped videos verified:', zip_count)
print('Decode cache (temporary Colab storage, NOT Drive):', VIDEO_CACHE)
print('Validated activity videos:', len(rows))


In [ ]:
# The data preflight has already checked real decoding for every selected video.
# Display frames from an actual decoded clip, not from the ZIP wrapper.
example = df.iloc[0]
example_video = DATA_ROOT / example.video_path
real_video = resolve_video_file(example_video)
print('Original Drive file:', example_video.name,
      '| ZIP-wrapped:', zipfile.is_zipfile(example_video))
print('Decoded local video:', real_video)
example_frames = sample_video(example_video, SEQ_LENGTH, IMAGE_SIZE)
assert example_frames.shape == (20, 64, 64, 3)
print('Example activity:', example.activity, '| tensor:', example_frames.shape,
      '| value range:', (float(example_frames.min()), float(example_frames.max())))
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for j, ax in enumerate(axes.flat):
    ax.imshow(example_frames[j*2]); ax.set_title(f'Frame {j*2+1}'); ax.axis('off')
plt.suptitle('Uniformly sampled frames: ' + example.activity)
plt.tight_layout()
plt.savefig(RESULTS_ROOT/'sampled_frames.png', dpi=150)
plt.show()


## 5. CNN–LSTM architecture and mathematics

**Block diagram**

```text
   Video (20 frames)
           │
           ▼
   RGB resize + scale
           │
           ▼
  TimeDistributed CNN ── Conv(32) ── Pool ── Conv(64) ── Pool ── Conv(128)
           │
           ▼
 Global Average Pooling per frame
           │  sequence of 20 feature vectors z_t
           ▼
       LSTM (128 units)
           │
           ▼
       Dropout (0.5)
           │
           ▼
       Dense (7) + softmax
           │
           ▼
        Activity label
```

**CNN** extracts spatial features \(z_t=f_\theta(X_t)\), shared across the 20 video frames. The **LSTM** learns temporal relationships:

\[
\begin{aligned}
i_t &= \sigma(W_i[z_t,h_{t-1}] + b_i), \\
f_t &= \sigma(W_f[z_t,h_{t-1}] + b_f), \\
\tilde c_t &= \tanh(W_c[z_t,h_{t-1}] + b_c), \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t, \\
o_t &= \sigma(W_o[z_t,h_{t-1}] + b_o), \\
h_t &= o_t \odot \tanh(c_t).
\end{aligned}
\]

The last hidden state produces class probabilities \(p(y=k\mid X)=\operatorname{softmax}_k(Wh_T+b)\). We minimize the average sparse categorical cross-entropy \(-\frac{1}{N}\sum_{n=1}^N\log p(y_n\mid X_n)\) with the Adam optimizer.

The project includes three scratch CNN–LSTM configurations corresponding to the PDF's architecture family, plus an **optional pretrained MobileNetV2–LSTM**. The output dimension is **seven**, not the older UCF101 model's 101.

In [ ]:
# Model variants: 3 scratch CNN–LSTM configurations and optional frozen MobileNetV2
"""TimeDistributed CNN feature extraction followed by sequence LSTM."""
from __future__ import annotations


def build_model(num_classes: int, variant: str = "cnn_lstm_3", seq_length: int = 20,
                image_size: int = 64, learning_rate: float = 1e-3):
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    if num_classes < 2:
        raise ValueError("At least two activity classes are required")
    inputs = layers.Input(shape=(seq_length, image_size, image_size, 3), name="video_rgb")
    x = inputs
    if variant == "mobilenetv2_lstm":
        backbone = tf.keras.applications.MobileNetV2(input_shape=(image_size, image_size, 3),
                                                      include_top=False, weights="imagenet", pooling="avg")
        backbone.trainable = False
        # Pretrained MobileNetV2 expects [-1,1], unlike scratch models.
        x = layers.Rescaling(2.0, offset=-1.0, name="mobilenet_input_range")(x)
        x = layers.TimeDistributed(backbone, name="pretrained_frame_cnn")(x)
        lstm_units = 128
    else:
        if variant == "cnn_lstm_1":
            filters = [32]
            flatten = True
            lstm_units = 64
        elif variant == "cnn_lstm_2":
            filters = [32]
            flatten = False
            lstm_units = 64
        elif variant == "cnn_lstm_3":
            filters = [32, 64, 128]
            flatten = False
            lstm_units = 128
        else:
            raise ValueError(f"Unknown model variant {variant!r}")
        for i, filt in enumerate(filters):
            x = layers.TimeDistributed(layers.Conv2D(filt, 3, padding="same", activation="relu"),
                                       name=f"frame_conv_{i+1}")(x)
            if i < 2:
                x = layers.TimeDistributed(layers.MaxPooling2D(2), name=f"frame_pool_{i+1}")(x)
        x = layers.TimeDistributed(layers.Flatten() if flatten else layers.GlobalAveragePooling2D(),
                                   name="frame_embedding")(x)
    x = layers.LSTM(lstm_units, name="temporal_lstm")(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(num_classes, activation="softmax", name="activity_probs")(x)
    model = Model(inputs, output, name=variant)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

## 6. Efficient training batches and strict evaluation separation

A Keras `Sequence` loads videos only for the current batch, limiting RAM usage. We shuffle **training** rows at every epoch and never shuffle the validation/test order. All checkpointing and model selection use validation loss; the final **test** labels are never used during training.

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
class_names = sorted(df.activity.unique().tolist())
label_to_index = {name: idx for idx, name in enumerate(class_names)}
df['label_id'] = df.activity.map(label_to_index)

class VideoBatch(tf.keras.utils.Sequence):
    def __init__(self, data, shuffle=False):
        super().__init__()
        self.data = data.reset_index(drop=True)
        self.shuffle = shuffle
        self.indices = np.arange(len(self.data))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.data) / BATCH_SIZE))

    def __getitem__(self, index):
        ids = self.indices[index*BATCH_SIZE:(index+1)*BATCH_SIZE]
        batch = self.data.iloc[ids]
        frames = np.stack([sample_video(DATA_ROOT/path, SEQ_LENGTH, IMAGE_SIZE)
                           for path in batch.video_path])
        labels = batch.label_id.to_numpy(dtype=np.int64)
        return frames, labels

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

train_data = VideoBatch(df[df.split=='train'], shuffle=True)
val_data = VideoBatch(df[df.split=='val'], shuffle=False)
print('Batch counts: train=', len(train_data), '| val=', len(val_data))
print('Classes (index order):', label_to_index)

## 7. Train and compare model configurations

Default: run **CNN–LSTM 3** for 25 epochs at most, with early stopping and the best validation-loss checkpoint. Expand `RUN_VARIANTS` in Section 1 to compare all three scratch architectures or the optional pretrained MobileNetV2 extension. Each experiment has its **own** output folder and `.keras` checkpoint.

**Important:** Runtime, training losses and accuracy must come from executed outputs. A notebook that has not been executed does not contain verified model performance.

In [ ]:
training_results = {}
for variant in RUN_VARIANTS:
    print('\n===== Training', variant, '=====')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    out = RESULTS_ROOT / variant
    out.mkdir(parents=True, exist_ok=True)
    model = build_model(len(class_names), variant, SEQ_LENGTH, IMAGE_SIZE)
    model.summary()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(out/'best.keras'), monitor='val_loss', save_best_only=True),
        tf.keras.callbacks.CSVLogger(str(out/'history.csv')),
    ]
    history = model.fit(train_data, validation_data=val_data,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    model.save(str(out/'final.keras'))
    metrics = {key: [float(v) for v in series] for key, series in history.history.items()}
    (out/'history.json').write_text(json.dumps(metrics, indent=2))
    (out/'class_names.json').write_text(json.dumps(class_names, indent=2))
    best_epoch = int(np.argmin(metrics['val_loss']))
    training_results[variant] = {
        'epochs_completed': len(metrics['loss']),
        'selected_epoch': best_epoch + 1,
        'best_val_loss': float(metrics['val_loss'][best_epoch]),
        'val_accuracy_at_best_loss_epoch': float(metrics['val_accuracy'][best_epoch]),
        'parameter_count': model.count_params(),
    }
    fig, axes = plt.subplots(1, 2, figsize=(13,4))
    for i, field in enumerate(('loss','accuracy')):
        axes[i].plot(metrics[field], label='train')
        axes[i].plot(metrics['val_'+field], label='validation')
        axes[i].set(xlabel='Epoch', ylabel=field.title(), title=f'{variant}: {field}')
        axes[i].legend()
    fig.tight_layout(); fig.savefig(out/'learning_curves.png', dpi=170); plt.show()
comparison = pd.DataFrame(training_results).T.sort_values('best_val_loss')
comparison.to_csv(RESULTS_ROOT/'validation_model_comparison.csv')
print(comparison)

## 8. Select best model on validation loss, then evaluate test once

The model with the **smallest validation loss** is selected. Only at this point do we run the 20-video test partition. It was not used for training, tuning or model selection.

Outputs: real test accuracy, per-class precision/recall/F1, confusion matrix, and CSV/JSON results. **Do not repeat test evaluation for tuning**; redo split from scratch if you need more research iterations.

In [ ]:
if not training_results:
    raise RuntimeError('Train at least one model in Section 7 first.')
selected_variant = min(training_results, key=lambda name: training_results[name]['best_val_loss'])
print('Selected variant by validation loss:', selected_variant)
selected_dir = RESULTS_ROOT / selected_variant
best_model = tf.keras.models.load_model(str(selected_dir/'best.keras'))
test_table = df[df.split=='test'].reset_index(drop=True)
test_data = VideoBatch(test_table, shuffle=False)
y_true = test_table.label_id.to_numpy(dtype=np.int64)
y_prob = best_model.predict(test_data, verbose=1)
y_pred = y_prob.argmax(axis=1)

report = classification_report(y_true, y_pred, labels=np.arange(len(class_names)),
                               target_names=class_names, zero_division=0, output_dict=True)
test_accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
print('Test accuracy:', round(test_accuracy, 4), '| Macro F1:', round(macro_f1, 4))
print(classification_report(y_true, y_pred, labels=np.arange(len(class_names)),
                            target_names=class_names, zero_division=0))
report['test_accuracy'] = test_accuracy
report['test_macro_f1'] = macro_f1
report['test_size'] = len(test_table)
(selected_dir/'test_metrics.json').write_text(json.dumps(report, indent=2))
comparison.loc[selected_variant, 'heldout_test_accuracy'] = test_accuracy
comparison.loc[selected_variant, 'heldout_test_macro_f1'] = macro_f1
comparison.to_csv(RESULTS_ROOT/'validation_model_comparison.csv')
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(selected_dir/'confusion_matrix.csv')
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.title('Held-out test confusion matrix: '+selected_variant)
plt.xlabel('Predicted class'); plt.ylabel('True class')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig(selected_dir/'confusion_matrix.png', dpi=180); plt.show()

## 9. Inference from a new video

Use the same frame sampling and normalization as training. The model outputs probabilities for **seven known activity classes**. A high softmax score is not proof of correctness on out-of-distribution activities.

In [ ]:
# Option A: set a Drive video path to predict on (default: one held-out test example).
PREDICTION_VIDEO = str(DATA_ROOT / test_table.iloc[0].video_path)
# Option B: to upload a new video, uncomment the following lines:
# from google.colab import files
# uploaded = files.upload()
# PREDICTION_VIDEO = next(iter(uploaded.keys()))

video_tensor = sample_video(PREDICTION_VIDEO, SEQ_LENGTH, IMAGE_SIZE)[None, ...]
probs = best_model.predict(video_tensor, verbose=0)[0]
predicted = class_names[int(np.argmax(probs))]
print('Video:', Path(PREDICTION_VIDEO).name)
print('Predicted activity:', predicted)
for label, prob in sorted(zip(class_names, probs), key=lambda x:-x[1]):
    print(f'{label:30s}: {prob:.3f}')
plt.figure(figsize=(8,4)); plt.barh(class_names, probs); plt.gca().invert_yaxis()
plt.xlim(0,1); plt.xlabel('Predicted probability'); plt.title('CNN–LSTM activity prediction')
plt.tight_layout(); plt.show()

## 10. Save experiment provenance and final deliverables

All artifacts go to `MyDrive/RAJ Rishav/HAR_Results/` so they remain available after Colab disconnects. Split references are written to `HAR_Kaggle_Subset/` and do not duplicate video bytes. Your repo contains the same split baseline.

In [ ]:
metadata = {
    'author': 'Raj Rishav',
    'dataset': 'sharjeelmazhar/human-activity-recognition-video-dataset',
    'dataset_root': str(DATA_ROOT),
    'video_count': len(df),
    'video_bytes': int(df.bytes.sum()),
    'seed': SEED,
    'split_counts': df.split.value_counts().to_dict(),
    'classes': class_names,
    'frames_per_video': SEQ_LENGTH,
    'frame_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'max_epochs': EPOCHS,
    'trained_variants': list(training_results),
    'selected_by': 'minimum validation loss',
    'selected_variant': selected_variant,
    'model_comparison': training_results,
    'final_test_accuracy': test_accuracy,
    'final_test_macro_f1': macro_f1,
    'tensorflow': tf.__version__,
    'python': platform.python_version(),
    'split_warning': 'Random video-level split; actor/group independence unknown',
}
(RESULTS_ROOT/'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
(RESULTS_ROOT/'class_names.json').write_text(json.dumps(class_names, indent=2))
print('Results saved to:', RESULTS_ROOT)
print('Selected checkpoint:', selected_dir/'best.keras')
print('Training class mapping:', RESULTS_ROOT/'class_names.json')

## 11. What to include in the submitted report

Write the **actual** dataset table (counts and bytes), the 7-class versus UCF101 distinction, uniform sampling equation, CNN–LSTM architecture diagram, LSTM update equations, dropout/optimizer/loss, stratified split, train–validation/test distinction, model parameter count and hyperparameters, epoch-wise training curves, validation comparison, true held-out test accuracy and macro-F1, class-wise report, confusion matrix and representative prediction. Cite Kaggle, the supplied project PDF and relevant CNN/LSTM/MobileNetV2 references. The report must not claim an unrun training result.

**Reproducibility checks:** fixed seed, no duplicate paths, all seven folders recognized distinctly, every class present in each subset, validation used for model selection, test held out until the final stage. For publication-grade results, obtain actor/recording group metadata and evaluate using group-separated clips rather than random video-level splits.